<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Lab 01 · The Importance of Return Tails for Investing and Trading

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the lab examples in a Colab-ready format so that you can
run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the lab text for detailed explanations and context.


## Project Setup
Set the project root so local data files, helper modules, and figure scripts
resolve correctly from `notebooks/labs/`.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks/labs"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Loading a Daily Price Series
The local file data/eod_data.csv contains several daily market series.


In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv(
    "../../data/eod_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)

In [ ]:
prices = data["SPY"].dropna()

In [ ]:
prices.head()

In [ ]:
returns = prices.pct_change().dropna()

In [ ]:
returns.describe().round(4)

## Identifying the Extreme Days
You can inspect the largest and smallest days directly.


In [ ]:
returns.nsmallest(5).round(4)

In [ ]:
returns.nlargest(5).round(4)

## Removing Only a Few Extreme Days
One of the clearest demonstrations of tail importance is to compare the full
sample with versions in which only the ten best or ten worst days are removed.


In [ ]:
def cumulative_path(rets):
    return (1.0 + rets).cumprod()

In [ ]:
def remove_extreme_days(rets, n=10, side="best"):
    out = rets.copy()
    if side == "best":
        idx = out.nlargest(n).index
    else:
        idx = out.nsmallest(n).index
    out.loc[idx] = 0.0
    return out

In [ ]:
full = cumulative_path(returns)

In [ ]:
no_best = cumulative_path(
    remove_extreme_days(returns, n=10, side="best")
)

In [ ]:
no_worst = cumulative_path(
    remove_extreme_days(returns, n=10, side="worst")
)

## Summarizing the Terminal Effect
You can also compare the terminal returns numerically.


In [ ]:
summary = pd.DataFrame(
    {
        "scenario": [
            "full_sample",
            "without_best_days",
            "without_worst_days",
        ],
        "terminal_return": [
            full.iloc[-1] - 1.0,
            no_best.iloc[-1] - 1.0,
            no_worst.iloc[-1] - 1.0,
        ],
    }
)

In [ ]:
summary.round(3)

## Measuring Concentration in Absolute Moves
The next question is whether this concentration changes through time.


In [ ]:
abs_rets = returns.abs()

In [ ]:
shares = []

In [ ]:
dates = []

In [ ]:
window = 252

In [ ]:
for end in range(window, len(abs_rets) + 1):
    sample = abs_rets.iloc[end - window : end]
    share = sample.nlargest(5).sum() / sample.sum()
    shares.append(float(share))
    dates.append(sample.index[-1])

In [ ]:
tail_share = pd.Series(shares, index=dates, name="tail_share")

## Figure Generation (Optional)
Run the lab figure scripts under `code/figures/` to regenerate the PNG files
under `assets/figures/`.


In [ ]:
# Figure generation code adapted from
# `code/figures/lab01_cumulative_paths_without_extremes.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab01_return_tails import cumulative_path
from code.labs.lab01_return_tails import load_returns
from code.labs.lab01_return_tails import remove_extreme_days

def main() -> None:
    """Generate cumulative paths for full, no-best, and no-worst returns."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    rets = load_returns("SPY")
    full = cumulative_path(rets)
    no_best = cumulative_path(remove_extreme_days(rets, n=10, side="best"))
    no_worst = cumulative_path(remove_extreme_days(rets, n=10, side="worst"))

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(full.index, full.values, label="Full sample", linewidth=1.5)
    ax.plot(no_best.index, no_best.values, label="Without 10 best days")
    ax.plot(no_worst.index, no_worst.values, label="Without 10 worst days")
    ax.set_title("Cumulative growth and the role of extreme days")
    ax.set_ylabel("Cumulative value (start = 1.0)")
    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from
# `code/figures/lab01_return_histogram_and_tails.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab01_return_tails import load_returns

def main() -> None:
    """Generate the return histogram and mark empirical tail cutoffs."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    rets = load_returns("SPY")
    q05 = rets.quantile(0.05)
    q95 = rets.quantile(0.95)

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.hist(rets, bins=40, color="tab:gray", edgecolor="black", alpha=0.8)
    ax.axvline(q05, color="tab:red", linestyle="--", label="5% tail cutoff")
    ax.axvline(q95, color="tab:green", linestyle="--", label="95% tail cutoff")
    ax.set_title("SPY daily return distribution with tail cutoffs")
    ax.set_xlabel("Simple daily return")
    ax.set_ylabel("Frequency")
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from
# `code/figures/lab01_rolling_tail_share.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab01_return_tails import load_returns
from code.labs.lab01_return_tails import rolling_tail_share

def main() -> None:
    """Generate the rolling tail-share figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    rets = load_returns("SPY")
    share = rolling_tail_share(rets, window=252)

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(share.index, share.values, color="tab:purple", linewidth=1.4)
    ax.set_title("Share of absolute annual move explained by five days")
    ax.set_ylabel("Share of absolute move")
    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.grid(True, linestyle="--", alpha=0.3)

    fig.tight_layout()

main()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
